# Phase 2 — Five learners and the first Qini comparison

Fits five uplift-related estimators on the Criteo training set, evaluates each with Qini curves on a shared held-out test set, and quantifies the gap between proper uplift models and a propensity-only strawman.

**The five learners:**

1. **Propensity baseline** — ranks by predicted P(convert), ignores treatment. The strawman from Blake/Nosko/Tadelis; included to quantify the loss.
2. **S-learner** — one model with treatment as an extra feature. Uplift = μ(x,1) − μ(x,0).
3. **T-learner** — two models, one per arm. Uplift = μ₁(x) − μ₀(x).
4. **Class Transformation** (Jaskowski & Jaroszewicz 2012) — recasts uplift as a single weighted binary classification.
5. **X-learner** (Künzel et al. 2019) — two-stage estimator with imputed counterfactuals; designed for imbalanced allocation.

Causal Forest is deferred to Phase 3 (needs a training-set downsample for memory reasons).

**Base learner.** All five wrap `HistGradientBoostingClassifier` / `Regressor` from scikit-learn — same algorithmic family as LightGBM (histogram-based GBDT), but with OpenMP bundled in the sklearn macOS wheel, so no `brew install libomp` dependency.

**Outcome.** `conversion`, ~0.3% base rate. Rare but the Phase 1 ATE is statistically real (+0.115 pp, 95% CI [+0.108, +0.122]).

In [ ]:
import sys
import time
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.uplift.data import load_criteo, FEATURE_COLS
from src.uplift.split import make_split
from src.uplift.evaluation import qini_curve, qini_coefficient
from src.uplift.plots import plot_qini_curves
from src.uplift.learners import (
    PropensityBaseline,
    SLearner,
    TLearner,
    ClassTransformationLearner,
    XLearner,
)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")

## 1. Load and split

80/20 train/test split, stratified by treatment so both arms are represented in the same proportion in each subset. Seed pinned to 42, so every learner sees the same train rows and gets scored on the same test rows — that's what makes the Qini comparison honest.

In [ ]:
df = load_criteo()
train, test = make_split(df)

print(f"train: {len(train):>10,} rows  ({train['treatment'].mean():.1%} treated)")
print(f"test:  {len(test):>10,} rows  ({test['treatment'].mean():.1%} treated)")
print(f"conversion base rate — train: {train['conversion'].mean():.4%}, test: {test['conversion'].mean():.4%}")

## 2. X, T, Y arrays

All five learners take the same three inputs: feature matrix `X`, binary treatment `T`, binary outcome `Y`. Outcome is `conversion` — visits are a leading indicator but conversions carry the revenue.

In [ ]:
X_tr = train[FEATURE_COLS].to_numpy()
T_tr = train["treatment"].to_numpy()
Y_tr = train["conversion"].to_numpy()

X_te = test[FEATURE_COLS].to_numpy()
T_te = test["treatment"].to_numpy()
Y_te = test["conversion"].to_numpy()

print(f"X_tr: {X_tr.shape}  X_te: {X_te.shape}")
print(f"Y_tr conversions: {Y_tr.sum():,}   Y_te conversions: {Y_te.sum():,}")

## 3. Fit all five learners

Every learner exposes `.fit(X, T, Y)` and `.predict_uplift(X)`. Uniform API is what makes the six-way comparison apples-to-apples.

**Expected fit time (M2 laptop):**

| model | fits inside | rough time |
|---|---|---|
| Propensity baseline | 1 model on 11M rows | ~1 min |
| S-learner | 1 model on 11M rows | ~1 min |
| T-learner | 2 models (1.7M control, 9.5M treated) | ~1 min |
| Class Transformation | 1 weighted model on 11M | ~1 min |
| X-learner | 5 models (2 outcome, 2 pseudo-uplift, 1 propensity) | ~3 min |

Total ~5–8 min.

In [ ]:
models = {
    "Propensity (baseline)": PropensityBaseline(),
    "S-learner":              SLearner(),
    "T-learner":              TLearner(),
    "Class Transformation":   ClassTransformationLearner(),
    "X-learner":              XLearner(),
}

fit_times = {}
for name, m in models.items():
    t0 = time.time()
    m.fit(X_tr, T_tr, Y_tr)
    fit_times[name] = time.time() - t0
    print(f"  {name:<25s} fit in {fit_times[name]:6.1f}s")

## 4. Predict uplift on the held-out test set

Each learner produces a per-user uplift score, which feeds directly into the Qini curve:

$$Q(k) = n_t(k) - n_c(k) \cdot \frac{N_t(k)}{N_c(k)}$$

$n_t, n_c$ are cumulative conversions among treated and control in the top-$k$ ranked users; $N_t, N_c$ are cumulative counts. Q(k) reads as "incremental conversions from targeting the top $k$."

In [ ]:
curves = {}
coefs = {}
for name, m in models.items():
    u = m.predict_uplift(X_te)
    curve = qini_curve(Y_te, T_te, u)
    curves[name] = curve
    coefs[name] = qini_coefficient(curve)

coef_df = pd.DataFrame({
    "model": list(coefs.keys()),
    "qini_coefficient": list(coefs.values()),
    "fit_time_s": [fit_times[n] for n in coefs.keys()],
}).sort_values("qini_coefficient", ascending=False).reset_index(drop=True)

coef_df.style.format({"qini_coefficient": "{:+.2f}", "fit_time_s": "{:.1f}"}) \
    .bar(subset=["qini_coefficient"], color="#b6532c", align=0)

## 5. Qini comparison

The Qini curve maps "share of population targeted" to "cumulative incremental conversions." Reading key:

- **Above the dashed random-targeting line** — model concentrates persuadables near the top of its ranking.
- **On the line** — no better than random.
- **Below the line** — anti-ranking; targets sleeping dogs or lost causes first.

The Blake/Diemert prediction is that uplift-native learners (X, T, Class Transformation) beat the propensity baseline visibly. This plot is where that prediction gets tested.

In [ ]:
# Order curves so the highest Qini coefficient plots on top with the accent color
ordered = dict(sorted(curves.items(), key=lambda kv: -coefs[kv[0]]))

fig, ax = plt.subplots(figsize=(8.5, 5.5))
plot_qini_curves(ordered, ax=ax, title="Qini curves — Criteo conversion, held-out 2.8M test set")
plt.tight_layout()
plt.show()

## 6. Top-K targeting

The Qini coefficient collapses everything into one number. The actionable version of the question is: *at a fixed targeting budget of N% of users, how many incremental conversions land?*

The table reports incremental conversions at 10%, 20%, and 50% targeting for each model, plus the fraction of total-population lift each represents.

In [ ]:
def top_k_lift(curve: pd.DataFrame, k_frac: float) -> float:
    """Cumulative incremental conversions when targeting top k_frac of users."""
    idx = (curve["share"] >= k_frac).idxmax()
    return float(curve["qini"].iloc[idx])

total_qini = curves["S-learner"]["qini"].iloc[-1]  # same for all models

rows = []
for name, curve in curves.items():
    row = {"model": name}
    for frac in [0.10, 0.20, 0.50]:
        row[f"lift@{int(frac*100)}%"] = top_k_lift(curve, frac)
        row[f"share@{int(frac*100)}%"] = top_k_lift(curve, frac) / total_qini
    rows.append(row)

topk_df = pd.DataFrame(rows).set_index("model")
topk_df = topk_df.reindex(coef_df["model"])  # order by qini coefficient

styled = topk_df.style.format({
    "lift@10%": "{:+.1f}", "lift@20%": "{:+.1f}", "lift@50%": "{:+.1f}",
    "share@10%": "{:.1%}", "share@20%": "{:.1%}", "share@50%": "{:.1%}",
})
styled

## 7. Reading the plot

The Qini coefficient table and curves above rank all five learners on the same test data. Three things the plot decides:

- **Propensity vs. the uplift-native learners.** Blake/Nosko/Tadelis predict a real gap. If Propensity ranks competitively, that's a data-specific finding worth naming in the writeup, not a defect to smooth over.
- **X-learner vs. T-learner.** X-learner is designed for imbalanced allocation, and Criteo is 85/15. The margin between the two is the cleanest test of that design intuition on this dataset.
- **Class Transformation.** IPW amplifies errors on the small arm. When the model lands near the top, theory holds. When it lands near the bottom, the weighting variance is usually the reason.

Phase 3 adds the Causal Forest on a 500k training downsample, evaluated on this same 2.8M test set. Phase 4 attaches bootstrap CIs to every Qini coefficient so the ranking claims survive noise.